# NLP Mastery Journey — Module 3: Text Representation

Welcome back. In **Module 1** you learned to *acquire* text. (Module 2 — cleaning/preprocessing — is assumed done: by the time text reaches these techniques, it's typically lowercased, tokenized, and stripped of noise.) Now: how do you turn text into **numbers** a model can actually use? That's what this notebook covers, end to end.

### What this notebook teaches
| # | Technique | Why it still matters |
|---|-----------|------------------------|
| 1 | Bag of Words (BoW) | The simplest, most interpretable baseline — still used for quick baselines & spam filters |
| 2 | N-grams (unigram/bigram/trigram) | Captures local word order BoW alone throws away |
| 3 | TF-IDF | The industry-standard classical representation for search & text classification |
| 4 | Feature Hashing | How production systems handle vocabularies too large to fit in memory |
| 5 | Co-occurrence matrices | The bridge concept between counting and embeddings |
| 6 | Word embeddings (Word2Vec/GloVe/FastText) | Dense vectors that capture meaning, not just counts |
| 7 | Contextual embeddings (BERT-style) preview | The modern default for anything accuracy-critical |
| 8 | Choosing + productionizing | A decision guide, and a save/load `Pipeline` template |

### How to use this notebook
- Every code cell is **commented line by line** — read the "why", not just the "what".
- **🔀 Alternatives** callouts tell you what else exists and when to reach for it.
- **📋 Copy-paste template** cells are written generically — reuse them directly in your own projects.
- Small, in-memory example sentences are used everywhere so **every cell in Parts 1–5 actually runs here**, with no internet or external files needed (unlike Module 1's live-scraping/API cells).


## 0. Setup

In [ ]:
# %pip install scikit-learn nltk gensim scipy numpy joblib

import numpy as np
import pandas as pd

print("Setup note: uncomment the pip install line above the first time you run this.")


## Part 1 — Bag of Words (BoW)

**Core idea**: represent a document as a vector counting how many times each vocabulary word appears — completely ignoring word order and grammar (hence "bag").

Example: `"the cat sat"` and `"sat the cat"` produce the **identical** BoW vector — that loss of order is BoW's main weakness (n-grams in Part 2 partially fix this).


In [ ]:
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are great pets",
]

# ── Step 1: build the vocabulary (every unique word across the corpus) ──────
# We do this "from scratch" once so the mechanics are crystal clear, THEN
# switch to sklearn (which does exactly this, just fast and battle-tested).

def build_vocabulary(documents):
    vocab = set()
    for doc in documents:
        for word in doc.lower().split():   # naive whitespace tokenizer for the demo
            vocab.add(word)
    return sorted(vocab)   # sorted() -> a stable, reproducible column order

vocab = build_vocabulary(corpus)
print(f"Vocabulary size: {len(vocab)}")
print(vocab)


In [ ]:
# ── Step 2: count word occurrences per document, from scratch ───────────────

def bow_vector(document, vocab):
    words = document.lower().split()
    counts = {word: words.count(word) for word in vocab}   # O(vocab * doc_len);
                                                             # fine for teaching,
                                                             # NOT how sklearn does it internally
    return [counts[word] for word in vocab]   # fixed order = the "vector"

manual_bow = [bow_vector(doc, vocab) for doc in corpus]
manual_bow_df = pd.DataFrame(manual_bow, columns=vocab)
manual_bow_df


In [ ]:
# ── Step 3: the real, production way — sklearn's CountVectorizer ────────────
# Same idea as above, but implemented efficiently with sparse matrices (won't
# blow up your RAM on a vocabulary of 100,000+ words like a dense array would).

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(corpus)   # fit_transform = learn vocab
                                                 # AND transform in one call

# bow_matrix is a SciPy sparse matrix — only non-zero entries are stored in
# memory. .toarray() below is just for VIEWING; never call it on a huge
# real corpus, or you'll materialize a dense matrix and run out of RAM.
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)
bow_df


In [ ]:
# 📋 COPY-PASTE TEMPLATE — the pattern you'll reuse in real projects:
#
# vectorizer = CountVectorizer(
#     lowercase=True,        # default True; set False if case is meaningful (e.g. "US" vs "us")
#     stop_words="english",  # drops common words like "the", "and" — or pass your own list
#     max_features=5000,     # cap vocabulary to the top-N most frequent words —
#                             # keeps the feature matrix a manageable size on large corpora
#     min_df=2,               # ignore words appearing in FEWER than 2 documents (likely typos/noise)
#     max_df=0.9,              # ignore words appearing in MORE than 90% of documents (too common to help)
# )
# X_train_bow = vectorizer.fit_transform(train_texts)   # fit ONLY on training data
# X_test_bow  = vectorizer.transform(test_texts)        # transform (not fit!) on test data
#
# ⚠️ Critical rule: fit_transform on train, transform (never fit again) on
# test/production data — fitting on test data leaks information and gives
# you an overly-optimistic evaluation.

print("Key CountVectorizer parameters shown above — these are the ones you'll tune in practice.")


In [ ]:
# ── Binary Bag of Words (presence/absence, not counts) ──────────────────────
# Sometimes whether a word appears at all matters more than how many times
# (e.g. spam detection: one occurrence of "viagra" is already a strong signal).

binary_vectorizer = CountVectorizer(binary=True)
binary_matrix = binary_vectorizer.fit_transform(corpus)
pd.DataFrame(binary_matrix.toarray(), columns=binary_vectorizer.get_feature_names_out())


## Part 2 — N-grams (Unigrams, Bigrams, Trigrams...)

An **n-gram** is a contiguous sequence of *n* tokens.
- **Unigram** (n=1): single words — this is what plain BoW above already used.
- **Bigram** (n=2): two-word sequences — e.g. `"not good"` (captures negation that unigrams destroy: `"not"` and `"good"` separately look positive!).
- **Trigram** (n=3): three-word sequences — captures more context, at the cost of a much bigger, sparser vocabulary.

This is the single biggest practical fix for BoW's "ignores word order" weakness.


In [ ]:
sentence = "the movie was not good at all"

# ── From scratch: generating n-grams manually ────────────────────────────────

def generate_ngrams(text, n):
    tokens = text.lower().split()
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    # tokens[i:i+n] slides a window of size n across the token list;
    # range stops at len-n+1 so the window never runs off the end

print("Unigrams:", generate_ngrams(sentence, 1))
print("Bigrams: ", generate_ngrams(sentence, 2))
print("Trigrams:", generate_ngrams(sentence, 3))

# Notice: "not good" appears as its OWN bigram feature — a unigram-only model
# would separately see "not" and "good" and lose that negation signal entirely.


In [ ]:
# ── NLTK's built-in ngrams function (same idea, standard library) ───────────
from nltk import ngrams
from nltk.tokenize import word_tokenize
import nltk
# nltk.download("punkt")   # run once if word_tokenize errors out

tokens = sentence.split()   # using .split() here to avoid needing the punkt
                             # download just for this demo; swap in
                             # word_tokenize(sentence) for real projects —
                             # it correctly splits punctuation, contractions, etc.

bigrams_list = list(ngrams(tokens, 2))
trigrams_list = list(ngrams(tokens, 3))
print("Bigrams (as tuples):", bigrams_list)
print("Trigrams (as tuples):", trigrams_list)


In [ ]:
# ── The industry way: CountVectorizer's ngram_range parameter ───────────────
# You almost never hand-roll n-gram generation in production — you just tell
# the vectorizer which range of n to include, and it handles vocabulary
# building, counting, and sparse storage for you.

ngram_vectorizer = CountVectorizer(ngram_range=(1, 2))   # (1,2) = unigrams AND bigrams together
ngram_matrix = ngram_vectorizer.fit_transform(corpus)

ngram_df = pd.DataFrame(ngram_matrix.toarray(), columns=ngram_vectorizer.get_feature_names_out())
ngram_df


In [ ]:
# 📋 COPY-PASTE TEMPLATE
#
# vectorizer = CountVectorizer(
#     ngram_range=(1, 3),     # unigrams + bigrams + trigrams, all combined into one vocabulary
#     min_df=3,               # n-gram vocab explodes fast — min_df/max_features
#     max_features=20000,     # become MUCH more important than with unigrams alone
# )
#
# 🔀 How far do you go?
# | ngram_range | Typical use                                                |
# |--------------|-------------------------------------------------------------|
# | (1, 1)        | Fast baselines, topic classification (word order matters less)|
# | (1, 2)         | Sentiment analysis — bigrams catch negation ("not good")     |
# | (1, 3)          | Short, phrase-heavy text (search queries, product titles)    |
# | (2, 2) only      | Rare — sometimes used for phrase-mining specifically          |
#
# ⚠️ Every extra n roughly squares your vocabulary size. Beyond trigrams,
# most teams switch to embeddings (Part 6) instead of pushing n further.

print("Guidance above — ngram_range is one of the highest-leverage hyperparameters to tune.")


## Part 3 — TF-IDF (Term Frequency – Inverse Document Frequency)

BoW's problem: very common words (like `"the"`) get huge counts but carry little meaning, while rare, distinctive words get buried. **TF-IDF** fixes this by *down-weighting* words that appear in many documents and *up-weighting* words that are frequent in one document but rare across the corpus.

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

- **TF (term frequency)**: how often term *t* appears in document *d* (often normalized by document length).
- **IDF (inverse document frequency)**: $\log\left(\frac{N}{1 + df(t)}\right)$ — *N* = total documents, *df(t)* = number of documents containing *t*. Rare terms get a HIGH IDF; terms in every document get an IDF near zero.


In [ ]:
# ── From scratch: computing TF-IDF manually, step by step ───────────────────
import math

def compute_tf(document, vocab):
    words = document.lower().split()
    return {word: words.count(word) / len(words) for word in vocab}
    # dividing by len(words) normalizes for document length, so a long
    # document doesn't automatically get bigger raw counts than a short one

def compute_idf(documents, vocab):
    n_docs = len(documents)
    idf = {}
    for word in vocab:
        # count how many documents contain this word at least once
        df = sum(1 for doc in documents if word in doc.lower().split())
        idf[word] = math.log(n_docs / (1 + df)) + 1   # "+1" smoothing avoids
                                                        # division by zero AND
                                                        # a zero IDF for words
                                                        # in every document
    return idf

idf_scores = compute_idf(corpus, vocab)
tf_scores_per_doc = [compute_tf(doc, vocab) for doc in corpus]

manual_tfidf = [
    [tf_scores_per_doc[i][word] * idf_scores[word] for word in vocab]
    for i in range(len(corpus))
]
pd.DataFrame(manual_tfidf, columns=vocab).round(3)


In [ ]:
# ── The production way: sklearn's TfidfVectorizer ───────────────────────────
# Does tokenization + counting + TF-IDF weighting + L2 normalization in one
# step (the exact IDF formula differs slightly in smoothing details from our
# from-scratch version above, but the CONCEPT is identical).

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
).round(3)
tfidf_df

# Notice "the" (appears in almost every document) gets a LOW score, while
# distinctive words like "pets" or "log" score higher — exactly the
# reweighting TF-IDF is designed to do.


In [ ]:
# ── TfidfTransformer: if you ALREADY have raw counts (e.g. from CountVectorizer) ─
# Useful when you want to inspect/manipulate raw counts first, then apply
# TF-IDF weighting as a separate step (TfidfVectorizer = CountVectorizer +
# TfidfTransformer fused into one convenient class).

from sklearn.feature_extraction.text import TfidfTransformer

count_vectorizer = CountVectorizer()
raw_counts = count_vectorizer.fit_transform(corpus)

tfidf_transformer = TfidfTransformer()
tfidf_from_counts = tfidf_transformer.fit_transform(raw_counts)

print("Same result as TfidfVectorizer, just split into two explicit steps:")
pd.DataFrame(tfidf_from_counts.toarray(), columns=count_vectorizer.get_feature_names_out()).round(3)


In [ ]:
# 📋 COPY-PASTE TEMPLATE — the TF-IDF setup you'll actually use in projects
#
# tfidf_vectorizer = TfidfVectorizer(
#     ngram_range=(1, 2),      # combine with n-grams (Part 2) — very common combo
#     stop_words="english",
#     max_features=10000,
#     min_df=2,
#     max_df=0.85,
#     sublinear_tf=True,        # uses 1 + log(tf) instead of raw tf — dampens
#                                # the effect of a word appearing 50 vs 5 times
#                                # (diminishing returns), a common production tweak
# )
# X_train = tfidf_vectorizer.fit_transform(train_texts)   # fit on train only
# X_test  = tfidf_vectorizer.transform(test_texts)         # transform only on test

print("This exact configuration is the classical baseline for text classification "
      "and search/retrieval (e.g. computing cosine similarity between TF-IDF vectors).")


In [ ]:
# ── TF-IDF's classic downstream use: cosine similarity search ───────────────
# This IS how many production keyword-search / "find similar documents"
# features work before anyone reaches for embeddings.

from sklearn.metrics.pairwise import cosine_similarity

query = ["cats sat on the mat"]
query_vec = tfidf_vectorizer.transform(query)          # transform, NOT fit_transform —
                                                          # the query must map onto the SAME
                                                          # vocabulary learned from the corpus
doc_vecs = tfidf_vectorizer.transform(corpus)

similarities = cosine_similarity(query_vec, doc_vecs)[0]
for doc, score in sorted(zip(corpus, similarities), key=lambda x: -x[1]):
    print(f"{score:.3f}  {doc}")


## Part 4 — Scaling Up: Feature Hashing & Co-occurrence Matrices

Two techniques that show up specifically once you're operating at **production/industry scale**, not just in a course notebook.


In [ ]:
# ── Feature Hashing (the "hashing trick") ────────────────────────────────────
# Problem CountVectorizer/TfidfVectorizer have at huge scale: they must keep
# an in-memory dict mapping every word -> a column index, and that dict grows
# forever as new vocabulary appears (e.g. streaming data, new products, typos).
#
# HashingVectorizer instead hashes each word directly to a fixed-size column
# index (e.g. always 2**18 columns) — NO vocabulary dictionary needed at all.
# Trade-off: hash collisions (two different words landing in the same column)
# are possible, and you can no longer map a column back to its original word.

from sklearn.feature_extraction.text import HashingVectorizer

hashing_vectorizer = HashingVectorizer(n_features=2**10, alternate_sign=False)
# n_features is fixed in advance and doesn't grow — this is exactly what makes
# HashingVectorizer usable in a streaming/online-learning production system
hashed_matrix = hashing_vectorizer.fit_transform(corpus)
print("Shape:", hashed_matrix.shape)   # always (n_docs, n_features), regardless of vocab size
print("Non-zero entries per doc:", hashed_matrix.getnnz(axis=1))

# 🔀 When to reach for HashingVectorizer specifically:
# - Streaming data where new vocabulary keeps appearing (can't pre-fit a vocab)
# - Extremely memory-constrained environments (no vocabulary dict to store)
# - You do NOT need to inspect/explain individual feature weights afterward


In [ ]:
# ── Co-occurrence matrix ─────────────────────────────────────────────────────
# Counts how often word pairs appear together within a fixed-size window.
# This is conceptually the SEED IDEA behind word embeddings (Part 6) — GloVe
# is literally trained by factorizing a giant co-occurrence matrix like this one.

def build_cooccurrence_matrix(documents, window_size=2):
    vocab = build_vocabulary(documents)
    idx = {word: i for i, word in enumerate(vocab)}
    matrix = np.zeros((len(vocab), len(vocab)))

    for doc in documents:
        tokens = doc.lower().split()
        for center_i, center_word in enumerate(tokens):
            # look at neighbors within `window_size` positions on either side
            start = max(0, center_i - window_size)
            end = min(len(tokens), center_i + window_size + 1)
            for neighbor_i in range(start, end):
                if neighbor_i != center_i:
                    matrix[idx[center_word], idx[tokens[neighbor_i]]] += 1
    return pd.DataFrame(matrix, index=vocab, columns=vocab)

cooc_df = build_cooccurrence_matrix(corpus, window_size=2)
cooc_df.astype(int)


## Part 5 — Word Embeddings (Word2Vec, GloVe, FastText) — Preview

Counting-based methods (BoW/TF-IDF) treat every word as an isolated column — `"good"` and `"great"` are just as unrelated as `"good"` and `"table"` to the model. **Word embeddings** fix this by learning a **dense, low-dimensional vector per word** (typically 100–300 numbers) such that words used in similar contexts end up with similar vectors — `"good"` and `"great"` land close together in that space.

This whole topic gets its own dedicated module later. Here's the landscape so you know the vocabulary and can start experimenting:

| Method | Trained on | Key idea |
|--------|------------|----------|
| **Word2Vec** | Your own corpus (or pretrained) | Predict a word from its context (CBOW) or context from a word (Skip-gram) |
| **GloVe** | Global co-occurrence statistics | Factorizes a co-occurrence matrix like the one you just built above |
| **FastText** | Character n-grams within words | Represents rare/misspelled/out-of-vocabulary words via subword pieces |


In [ ]:
# ── Training your OWN Word2Vec model on a small corpus (gensim) ─────────────
# Needs a reasonably large corpus to produce meaningful vectors — this tiny
# 3-sentence example is only here to show the API; treat the actual numbers
# as illustrative, not meaningful.

# from gensim.models import Word2Vec
#
# tokenized_corpus = [doc.lower().split() for doc in corpus]
#
# w2v_model = Word2Vec(
#     sentences=tokenized_corpus,
#     vector_size=50,     # dimensionality of each word vector
#     window=2,            # how many neighboring words count as "context"
#     min_count=1,          # ignore words appearing fewer than this many times
#                            # (1 here only because our demo corpus is tiny;
#                            # real projects typically use 5+)
#     sg=1,                  # sg=1 -> skip-gram, sg=0 -> CBOW
# )
#
# vector_for_cat = w2v_model.wv["cat"]              # a 50-dim numpy array
# similar_words = w2v_model.wv.most_similar("cat")   # nearest neighbors by cosine similarity

print("gensim Word2Vec training pattern shown above.")


In [ ]:
# ── Using PRETRAINED embeddings (far more common in practice than training
#    your own from scratch, unless you have a huge domain-specific corpus) ──

# import gensim.downloader as api
# glove_vectors = api.load("glove-wiki-gigaword-100")   # downloads once, ~130MB, then cached
# print(glove_vectors["cat"][:10])                       # first 10 of 100 dimensions
# print(glove_vectors.most_similar("cat"))
# print(glove_vectors.similarity("cat", "dog"))

print("Pretrained-embedding loading pattern shown above — needs internet the first run.")

# 🔀 Alternatives / extras
# - fasttext (Facebook's own library) -> best when your text has lots of typos,
#   rare words, or a morphologically rich language (handles OOV words gracefully)
# - spaCy's built-in vectors (nlp("cat").vector) -> convenient if you're
#   already using spaCy for tokenization/NER in the same pipeline


In [ ]:
# ── Turning a document into a SINGLE vector by averaging its word vectors ───
# The simplest way to go from per-word embeddings to a per-document feature
# vector usable by a classifier (a reasonable baseline before trying more
# sophisticated pooling or moving to sentence embeddings in Part 6).

# def average_embedding(document, embedding_model, dim=100):
#     words = document.lower().split()
#     vectors = [embedding_model[w] for w in words if w in embedding_model]
#     if not vectors:                          # no known words -> return a zero vector
#         return np.zeros(dim)
#     return np.mean(vectors, axis=0)
#
# doc_vector = average_embedding(corpus[0], glove_vectors, dim=100)

print("average_embedding() pattern shown above — a common, simple document-level baseline.")


## Part 6 — Contextual Embeddings (BERT-style) — A Look Ahead

One real limitation Word2Vec/GloVe share: each word gets exactly **one** fixed vector, regardless of context — `"bank"` gets the same vector in `"river bank"` and `"bank account"`. **Contextual embeddings** (BERT, RoBERTa, and friends) fix this by producing a *different* vector for the same word depending on the sentence it's in. This is the modern industry default whenever raw accuracy matters more than interpretability or latency.

Full transformer models are their own upcoming module — here's just enough to try it today.


In [ ]:
# ── Sentence embeddings with `sentence-transformers` ────────────────────────
# The most common "just give me a good vector for this text" production tool —
# built on top of BERT-family models, but pooled into one vector per sentence
# so you don't have to think about per-token outputs yourself.

# %pip install sentence-transformers
#
# from sentence_transformers import SentenceTransformer
#
# model = SentenceTransformer("all-MiniLM-L6-v2")   # small, fast, strong baseline model
# sentence_vectors = model.encode(corpus)             # shape: (n_docs, 384)
#
# from sklearn.metrics.pairwise import cosine_similarity
# similarity_matrix = cosine_similarity(sentence_vectors)
# print(similarity_matrix.round(2))

print("Sentence-embedding pattern shown above — needs internet the first run to download the model.")

# 🔀 Where this fits in vs. everything above
# | Representation        | Captures word order? | Captures meaning/synonyms? | Cost            |
# |--------------------------|------------------------|-------------------------------|-------------------|
# | BoW / TF-IDF               | No                       | No                              | Very cheap          |
# | + n-grams                   | Partially (local)         | No                                | Cheap                 |
# | Word2Vec / GloVe (averaged)  | No (averaging loses order)  | Yes (word-level)                   | Cheap once trained/loaded|
# | Contextual (BERT/sentence-transformers)| Yes                    | Yes (context-aware)                  | Most expensive (needs a GPU to be fast at scale)|


## Part 7 — Choosing the Right Representation, and Shipping It

### Decision guide

| Situation | Reach for... |
|-----------|--------------|
| Fast baseline, need results in an hour | TF-IDF (unigrams) + a linear model (LogisticRegression / LinearSVC) |
| Sentiment/negation matters | TF-IDF with `ngram_range=(1,2)` |
| Streaming data, vocabulary keeps growing | `HashingVectorizer` |
| Search / "find similar documents" | TF-IDF + cosine similarity, or sentence embeddings for better recall |
| Small labeled data, need semantic generalization | Pretrained word embeddings (GloVe/FastText), averaged |
| Accuracy is the top priority, latency budget allows it | Contextual embeddings (BERT/sentence-transformers) or fine-tuning a transformer directly |
| Explainability required (e.g. regulated industry) | TF-IDF/BoW — you can always point to exactly which words drove a prediction |


In [ ]:
# ── 📋 COPY-PASTE TEMPLATE: a full, production-shaped classification pipeline ─
# Bundles vectorizer + model into ONE object, so there's no risk of
# accidentally applying a different vectorizer at inference time than the one
# used at training time (a surprisingly common real-world bug).

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

labels = [1, 1, 0]   # toy labels lining up with `corpus`, just to make .fit() runnable

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, stop_words="english")),
    ("classifier", LogisticRegression(max_iter=1000)),
])

pipeline.fit(corpus, labels)
predictions = pipeline.predict(corpus)
print("Predictions:", predictions)

# At inference time you ONLY ever call pipeline.predict(new_texts) — the
# TF-IDF transform step is applied automatically and consistently, exactly
# as it was during training.


In [ ]:
# ── Saving and loading the fitted pipeline for production use ───────────────
import joblib

# joblib.dump(pipeline, "text_classifier_pipeline.joblib")
#
# # ... later, in a completely different process/server ...
# loaded_pipeline = joblib.load("text_classifier_pipeline.joblib")
# loaded_pipeline.predict(["a brand new sentence to classify"])

print("joblib save/load pattern shown above — this is how you ship a fitted "
      "vectorizer+model pair to production without retraining.")

# ⚠️ Production gotchas worth remembering:
# - Save the ENTIRE pipeline (vectorizer + model), never just the model —
#   otherwise inference-time text won't map onto the same vocabulary/weights.
# - Pin your scikit-learn version; unpickling a Pipeline saved with a
#   different sklearn version can silently break or refuse to load.
# - Log out-of-vocabulary rates over time in production — a rising OOV rate
#   signals your vocabulary is going stale and the model needs retraining.


## Recap & What's Next

You can now turn raw text into numeric features using every major classical technique, know exactly when each one is the right (or wrong) tool, and have copy-paste-ready templates for:
- `CountVectorizer` (BoW, binary BoW, n-grams)
- `TfidfVectorizer` / `TfidfTransformer` (+ cosine-similarity search)
- `HashingVectorizer` (streaming/production scale)
- A saved, production-shaped `Pipeline`

### Try this before the next lesson
1. Take the `corpus.jsonl` you built at the end of Module 1 and fit a `TfidfVectorizer(ngram_range=(1,2))` on it.
2. Pick two documents from it and compute their cosine similarity.
3. Wrap it in a `Pipeline` with `LogisticRegression`, fit it on some labeled examples, and save it with `joblib`.

### Next lesson in your NLP mastery path
**Module 4: Word & Sentence Embeddings, In Depth** — how Word2Vec/GloVe/FastText are actually trained under the hood, visualizing embedding spaces, and bridging into transformer-based contextual embeddings before you move on to modeling.
